# Activity (Advanced, Optional): Monte Carlo Tree Search
This activity is __advanced and optional__. The MCTS Watch-Demo showed that Monte Carlo Tree Search (MCTS) can find the same root action as exact value iteration without ever computing $V^{\star}(s)$ for every state, by building a small search tree rooted at the current state $s_{0}$ and guided by random rollouts and the UCT (Upper Confidence bound for Trees) selection rule. In this activity, you will run `mcts` yourself on the same deterministic $5\times5$ grid world (goal at $(5,5)$, reward $+100$; hazard at $(2,2)$, reward $-100$), and study how two knobs shape the action it returns: the UCT exploration constant $c$ and the search iteration budget.

> __Learning Objectives.__
>
> In this activity, you will run Monte Carlo tree search on a deterministic grid world, vary the UCT exploration constant and the iteration budget, and compare the resulting root action to the exact optimal action:
> * __Run MCTS from a state:__ use `mcts` to search from a root state $s_{0}$ and return its estimated best action.
> * __Vary the exploration constant $c$ and the iteration budget:__ sweep the UCT exploration constant $c$ at a fixed iteration budget, and sweep the iteration budget at a fixed $c$, recording how often the root action returned by `mcts` matches $\pi^{\star}(s_{0})$.
> * __Observe convergence to the optimal action:__ compare the root action from `mcts` to $\pi^{\star}(s_{0})$ from value iteration as the iteration budget grows, and see MCTS converge to the optimal action.

Let's get started.
___


## Theory
MCTS estimates action values at a single root state $s_{0}$ by repeatedly building and refining a search tree, instead of solving the Bellman equations for every state. Each iteration of `mcts` runs four phases — selection, expansion, simulation (rollout), and backpropagation (Watch-Demo, Theory section). Selection descends the tree using the UCT rule
$$
\mathrm{UCT}(s,a) = Q(s,a) + c\sqrt{\frac{\ln N(s)}{N(s,a)}},
$$
where $Q(s,a)$ is the current mean-return estimate for action $a$ at state $s$, $N(s)$ is the number of times $s$ has been visited, $N(s,a)$ is the number of times action $a$ has been chosen at $s$, and $c>0$ is the exploration constant. The first term favors actions with high estimated value (exploitation); the second term grows for actions tried rarely relative to $N(s)$ (exploration). After `model.iterations` repetitions of the four phases, `mcts(mdp, model, s0; rng)` returns $\arg\max_{a}Q(s_{0},a)$, the single best action at the root, and discards the rest of the tree.

This activity studies how two knobs shape that returned action:
* __The exploration constant $c$__ sets the balance between exploitation and exploration in the UCT rule. A $c$ that is too small lets the search commit early to whichever action's first few rollouts looked best; a $c$ that is too large keeps spreading a limited budget across all actions instead of concentrating simulations on the best one.
* __The iteration budget__ `model.iterations` sets how many times the four phases repeat before the search stops. A larger budget gives the tree more visits to sharpen its $Q(s,a)$ estimates and narrow the UCT exploration bonus, at the cost of more computation.


## Setup
This activity uses functions defined in the `src` directory and a small set of external packages. The `include(...)` call below runs `Include.jl`, which activates the local project environment, loads the packages, and includes our code. The first run may take a few minutes while packages are installed and precompiled.


In [1]:
include("Include.jl");

  Activating 

project at `~/Desktop/julia_work/CHEME-140-eCornell-Repository/courses/CHEME-145/module-4`


## Build the Grid World and the Exact Solution (Reference)
We reuse the $5\times5$ grid world with one goal cell $(5,5)$ (reward $+100$) and one hazard cell $(2,2)$ (reward $-100$); both are absorbing. As in the MCTS Watch-Demo, this world is built __deterministically__ (no slip): the chosen action's move is always carried out, so any randomness in the results below comes only from MCTS's own random rollouts, not from the environment. If a move would carry the agent off the grid, the agent instead stays in its current cell and pays `offgrid_penalty=-1.0`, which also bounds the return of a random rollout that repeatedly bumps a wall. We solve the resulting MDP exactly with value iteration to obtain $V^{\star}$ and the optimal policy $\pi^{\star}$; this is the ground-truth reference the MCTS root actions below are compared against.


In [2]:
rewards = Dict{Tuple{Int,Int},Float64}((5,5)=>100.0, (2,2)=>-100.0);
world = build(MyRectangularGridWorldModel, (nrows=5, ncols=5, rewards=rewards));
absorbing = Set(keys(rewards));
mdp = build_mdp(world, 0.95; step_reward=-1.0, offgrid_penalty=-1.0, absorbing=absorbing);
sol = solve(build(MyValueIterationModel, (maxiterations=10_000, ϵ=1e-9)), mdp);
π_star = policy(Q(mdp, sol.V));

## Experiment 1: The Exploration Constant $c$
We start MCTS from $(1,2)$ — several moves from the goal, and off the grid's diagonal symmetry axis, so it has a single best action rather than a tie between two directions — and fix a moderate iteration budget of `iterations = 150`: large enough that a well-tuned $c$ can find the correct root action almost every time, but small enough that a poorly tuned $c$ still fails. For each of $c \in \{1, 10, 50, 200\}$, we rerun `mcts` with 20 different random seeds and count how often the returned root action matches $\pi^{\star}(s_{0})$. A small $c$ under-explores: the search commits early to whichever action's first rollouts looked best and rarely revisits the others. A large $c$ over-explores: the UCT bonus term dominates the score, so the search keeps spreading its limited budget across all four actions instead of concentrating simulations on the best one. A moderate $c$ balances the two and should find the correct root action most often.

__Try changing it:__ after running the cell below, change `iterations` (e.g., to `2_000`) and re-run — with a large enough budget every $c$ eventually finds the correct action, and the gap between them shrinks.


In [3]:
let
    s0 = world.states[(1,2)];
    for c_try ∈ (1.0, 10.0, 50.0, 200.0)
        hits = count(1:20) do trial
            model = build(MyMCTSModel, (iterations = 150, c = c_try, horizon = 100, depth = 20));
            mcts(mdp, model, s0; rng = Random.MersenneTwister(trial)) == π_star[s0];
        end
        println("c = $(c_try):  correct root action in ", hits, "/20 runs");
    end
end

c = 1.0:  correct root action in 0

/20 runs
c = 10.0:  correct root action in 1/20 runs
c = 50.0:  correct root action in 15/20 runs
c = 200.0:  correct root action in 7/20 runs


## Experiment 2: Convergence vs. Iteration Budget
State $(1,2)$ is far enough from the goal that a small MCTS budget is not always enough to reliably find the optimal root action, but as the iteration budget grows, the search tree and its rollout-based value estimates should converge on the correct choice. Fixing $c = 50$ (the best-performing value from Experiment 1) and a longer rollout horizon and tree depth (`horizon = 200`, `depth = 30`) so the search can still reach the goal at larger budgets, we run `mcts` from $(1,2)$ at three iteration budgets with a fixed random seed and compare the returned action to $\pi^{\star}(s_{0})$.

__Try changing it:__ swap `(1,2)` for another off-diagonal start state, or change the seed in `Random.MersenneTwister(5)`, and re-run to see whether the same iteration budget is still enough to reach the optimal action.


In [4]:
let
    s0 = world.states[(1,2)];
    for iters ∈ (100, 1_000, 5_000)
        model = build(MyMCTSModel, (iterations = iters, c = 50.0, horizon = 200, depth = 30));
        a = mcts(mdp, model, s0; rng = Random.MersenneTwister(5));
        println("iterations = $(iters):  MCTS action = ", a, "   optimal = ", π_star[s0]);
    end
end

iterations = 100:  MCTS action = 1   optimal = 4
iterations = 1000:  MCTS action = 

4   optimal = 4
iterations = 5000:  MCTS action = 4   optimal = 4


## Summary
This activity ran Monte Carlo tree search on the deterministic $5\times5$ grid world from the Watch-Demo, sweeping the UCT exploration constant $c$ at a fixed iteration budget and then increasing the iteration budget at a fixed $c$, and compared the resulting root action to the exact optimal action $\pi^{\star}(s_{0})$ from value iteration.

> __Key Takeaways:__
>
> * **The exploration constant $c$ controls the exploration/exploitation balance:** at $(1,2)$ with a $150$-iteration budget, a small $c$ under-explored and a large $c$ over-explored, while a moderate $c$ found the correct root action most often of the 20 seeds tested.
> * **Harder start states need a larger iteration budget:** at $(1,2)$, the root action returned by `mcts` did not match $\pi^{\star}(s_{0})$ at a small iteration budget but matched it once the budget was large enough, and stayed correct as the budget grew further.
> * **MCTS approximates the optimum without full dynamic programming:** unlike value iteration, which sweeps every state and action against the full transition model $P(s^{\prime}\mid s,a)$, `mcts` only searches from the current root state, yet its returned action converged to the same action as $\pi^{\star}(s_{0})$ given a large enough $c$ and iteration budget.

MCTS trades the exactness and full-state coverage of value iteration for a search that only needs a simulator and a budget of iterations to choose a good action from the current state, with the exploration constant $c$ and the iteration budget together controlling how reliably that action matches the true optimum.
___


### Additional Resources
* Kocsis, L., & Szepesvári, C. (2006). Bandit Based Monte-Carlo Planning. In _European Conference on Machine Learning (ECML)_, pp. 282–293.
* Browne, C. B., Powley, E., Whitehouse, D., Lucas, S. M., Cowling, P. I., Rohlfshagen, P., Tavener, S., Perez, D., Samothrakis, S., & Colton, S. (2012). A Survey of Monte Carlo Tree Search Methods. _IEEE Transactions on Computational Intelligence and AI in Games_, 4(1), 1–43.
* Sutton, R. S., & Barto, A. G. (2018). _Reinforcement Learning: An Introduction_ (2nd ed.), Chapter 8. MIT Press.
